# 02 · Campaign — curate the labeled dataset + predict everything

**Standard slot:** *design campaign.* **For Project 01 this means:** there is nothing to *generate* —
your "campaign" is assembling a trustworthy **labeled benchmark** and predicting every item with all
three tools (D2). The labels are the hard part; start dataset curation in Week 1.

## Setup paths

In [ ]:
import sys, os
# Make the project's scripts/ and the cohort's shared/ importable.
# Adjust these if your Colab working directory differs (see 00_setup §5 for Drive mounting).
sys.path.insert(0, os.path.abspath("../scripts"))
sys.path.insert(0, os.path.abspath("../../../shared"))
os.makedirs("results", exist_ok=True)
print("paths ready; cwd =", os.getcwd())

## 1 · The dataset schema

Create `data/dataset.csv` with one row per labeled item. Provenance columns are **graded** — every
item must say where its label came from. See `data/README.md` for sources and the licensing policy.

In [ ]:
import pandas as pd, os

SCHEMA = ["id", "sequence", "design_type", "source_doi", "source_table",
          "license", "outcome", "outcome_detail", "notes"]

# A tiny seed so the notebook runs end-to-end before your real curation is done.
# REPLACE with your ≥40 designs + ≥10 natural refs. outcome ∈ {success, fail}.
seed_rows = [
    dict(id="nat_ubiquitin", sequence=("MQIFVKTLTGKTITLEVEPSDTIENVKAKIQDKEGIPPDQQRLIFAGKQLEDGRTLSDYNIQKESTLHLVLRLRGG"),
         design_type="monomer", source_doi="PDB:1UBQ", source_table="RCSB",
         license="public-domain", outcome="success", outcome_detail="natural, folds", notes="positive ref"),
    dict(id="nat_trpcage", sequence="NLYIQWLKDGGPSSGRPPPS", design_type="monomer",
         source_doi="PDB:1L2Y", source_table="RCSB", license="public-domain",
         outcome="success", outcome_detail="natural mini-protein", notes="positive ref"),
]
os.makedirs("../data", exist_ok=True)
path = "../data/dataset.csv"
if not os.path.exists(path):
    pd.DataFrame(seed_rows, columns=SCHEMA).to_csv(path, index=False)
    print("seeded", path, "— REPLACE with your curated set (>=40 designs + >=10 refs).")
df = pd.read_csv(path)
print(df.shape); df.head()

## 2 · Predict every item with every tool

Embarrassingly parallel but compute-bound on a T4: **triage with ESMFold** (seconds), reserve AF2
full-MSA for the subset that matters, batch overnight. Below uses the `mock` backend so the loop
runs anywhere; switch `TOOLS` to the real ones on Colab.

In [ ]:
from predict import predict
import pandas as pd, time

TOOLS = ["mock"]   # → ["esmfold", "af2", "boltz"] on Colab once installed
rows = []
for _, r in df.iterrows():
    for tool in TOOLS:
        p = predict(r["sequence"], tool=tool)
        rows.append(dict(id=r["id"], design_type=r["design_type"], tool=p.tool,
                         plddt=p.plddt, pae=p.pae, ptm=p.ptm,
                         runtime_s=p.runtime_s, ok=p.ok, error=p.error,
                         outcome=r["outcome"]))
pred = pd.DataFrame(rows)
pred.to_csv("results/predictions.csv", index=False)
print("wrote results/predictions.csv", pred.shape)
pred.head()

## 3 · MSA-depth ablation (AF2) `[extension]`

On a subset, run AF2 with full MSA vs single-sequence and compare pLDDT/scRMSD. This shows how much
of AF2's confidence is "borrowed" from homologs — directly relevant to how it behaves on *de novo*
sequences with no MSA. Record both settings per item.

In [ ]:
# Pseudocode for the ablation (fill in once ColabFold is wired in predict.py):
# for seq in subset:
#     full = predict(seq, tool="af2")              # default mmseqs2 MSA
#     single = predict(seq, tool="af2", msa_mode="single_sequence")
#     record (seq, full.plddt, single.plddt, delta)
print("Ablation scaffold — implement with the real AF2 backend.")

## D2 checklist
- [ ] `data/dataset.csv`: ≥40 designs + ≥10 natural refs, full provenance per row.
- [ ] `results/predictions.csv`: every item × tool with parsed metrics.
- [ ] Prediction log (tool versions, params, seeds, runtimes) in `LOG.md`.
- [ ] 3–4 page interim report.

**Next:** `03_filter_and_rank.ipynb` — run the shared filter on your predictions.